In [18]:
import os
import pandas as pd
import re

In [16]:
#
# 1. Load the local data
#
df = pd \
    .read_csv( '../../data/preschools/ccld-reports-capacities.csv' ) \
    .assign( dt = lambda _df: pd.to_datetime( _df['report_date'] ) ) \
    [lambda _df: ~pd.isnull( _df['total_enrolled'] )] \
    .assign( dt_rank = lambda _df: _df.groupby( 'facility_id' )['dt'].rank( method='first', ascending = False ) ) \
    [lambda _df: _df['dt_rank'] == 1 ] \
    .merge( pd.read_csv( '../../data/preschools/ccld-details-full.csv' )[['facility_id','facility_name','detail_url']] ) \
    [['facility_id','facility_name','report_date','capacity','total_enrolled','detail_url']]

In [18]:
df.to_csv( '~/Downloads/preschool-capacities-latest.csv', index=False )

In [23]:
#
# Now let's figure out enrollment trends
#
school_df = pd.read_csv( '../../data/school-mapping/merged/school-id-mapping.csv' )


enrollment_df = pd.DataFrame()

data_dir = '../../data/enrollment/raw/'
for filename in os.listdir( data_dir ):
    if not re.search( r'\d{4}.*\.txt$', filename ):
        continue

    tmp = pd.read_csv( f"{data_dir}/{filename}", sep = '\t', encoding = 'latin-1' )
    code_col = [ col for col in tmp if re.search( r'CODE', col.upper() ) ][0]
    tmp = tmp.merge( school_df, right_on = 'CDSCode', left_on = code_col )

    enrollment_df = pd.concat( [ enrollment_df, tmp ], axis = 0 )

enr198192-v2.txt
enr199397-v2.txt
cdenroll2324-v2.txt
enr201719-v2.txt
enr201416-v2.txt
enr200713-v3.txt


/var/folders/lr/v7klkd8j671bd11k9l_nvwcm0000gn/T/ipykernel_48694/483107671.py:15: DtypeWarning: Columns (21) have mixed types. Specify dtype option on import or set low_memory=False.
  tmp = pd.read_csv( f"{data_dir}/{filename}", sep = '\t', encoding = 'latin-1' )


enr202022-v2.txt
cdenroll2425.txt


/var/folders/lr/v7klkd8j671bd11k9l_nvwcm0000gn/T/ipykernel_48694/483107671.py:15: DtypeWarning: Columns (7,11,12,13,14,15,16,17,18,19,20,21,22,23,24) have mixed types. Specify dtype option on import or set low_memory=False.
  tmp = pd.read_csv( f"{data_dir}/{filename}", sep = '\t', encoding = 'latin-1' )


In [39]:
enrollment_ratio_df = enrollment_df \
    .assign( enrollment = lambda _df: _df['ENR_TOTAL'].fillna( 0 ) + _df['TOTAL_ENR'].fillna( 0 ) ) \
    [['ACADEMIC_YEAR','CDS_CODE','enrollment']] \
    .groupby( [ 'ACADEMIC_YEAR', 'CDS_CODE' ], as_index = False ) \
    .sum() \
    .assign( enrollment_max = lambda _df: _df.groupby( 'CDS_CODE' )['enrollment'].transform( max ) ) \
    [lambda _df: _df['ACADEMIC_YEAR'] == '2022-23'] \
    .assign( enrollment_ratio = lambda _df: _df['enrollment'] / _df['enrollment_max'] ) \
    .sort_values( 'enrollment_ratio' ) \
    .merge( school_df[['CDSCode','School','school_id']], left_on = 'CDS_CODE', right_on = 'CDSCode' ) \
    .head( 30 )

/var/folders/lr/v7klkd8j671bd11k9l_nvwcm0000gn/T/ipykernel_48694/1686041903.py:6: FutureWarning: The provided callable <built-in function max> is currently using SeriesGroupBy.max. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "max" instead.
  .assign( enrollment_max = lambda _df: _df.groupby( 'CDS_CODE' )['enrollment'].transform( max ) ) \


In [42]:
nearest_preschool_df = pd.read_csv( './nearest_preschools.csv' )
nearest_preschool_df \
    .merge( enrollment_ratio_df[['school_id','enrollment','enrollment_max','enrollment_ratio']], on = 'school_id' ) \
    .rename( columns = { 'enrollment': 'ES_enrollment_ay2223', 'enrollment_max': 'ES_enrollment_max', 'enrollment_ratio': 'ES_enrollment_ratio_ay2223' } ) \
    .to_csv( 'nearest_preschools_with_enrollment_ratio.csv', index = False )

In [33]:
school_df.columns

Index(['Unnamed: 0', 'CDSCode', 'NCESDist', 'NCESSchool', 'StatusType',
       'County', 'District', 'School', 'Street', 'StreetAbr', 'City', 'Zip',
       'State', 'MailStreet', 'MailStrAbr', 'MailCity', 'MailZip', 'MailState',
       'Phone', 'Ext', 'FaxNumber', 'WebSite', 'OpenDate', 'ClosedDate',
       'Charter', 'CharterNum', 'FundingType', 'DOC', 'DOCType', 'SOC',
       'SOCType', 'EdOpsCode', 'EdOpsName', 'EILCode', 'EILName', 'GSoffered',
       'GSserved', 'Virtual', 'Magnet', 'YearRoundYN', 'FederalDFCDistrictID',
       'Latitude', 'Longitude', 'AdmFName', 'AdmLName', 'LastUpDate',
       'Multilingual', 'school_modified', 'school_id', 'school_name'],
      dtype='object')